In [12]:
from hci import lib
import ctypes
import numpy as np
import math

rng = np.random.default_rng()
libhci = lib.load_library("libhci")

In [3]:
norb = 8
nocc = 4
combs = np.empty(min(nocc, norb-nocc)+1, dtype=np.uint64)
inverted = libhci.get_consecutive_combs(combs.ctypes.data_as(ctypes.c_void_p), 
                 ctypes.c_size_t(norb),
                 ctypes.c_size_t(nocc))
print(inverted, combs)

0 [ 1  8 28 56 70]


In [4]:
norb = 8
nocc = 4
rank_table = np.empty((nocc, norb-nocc+1), dtype=np.uint64, order='C')
libhci.get_rank_table(rank_table.ctypes.data_as(ctypes.c_void_p),
                                       ctypes.c_size_t(norb),
                                       ctypes.c_size_t(nocc))
print(rank_table)

[[ 0  1  2  3  4]
 [ 0  1  3  6 10]
 [ 0  1  4 10 20]
 [ 0  1  5 15 35]]


In [5]:
rank = 0
index = libhci.find_row_index(ctypes.c_uint64(rank),
                              rank_table[2][:].ctypes.data_as(ctypes.c_void_p),
                              ctypes.c_size_t(norb),
                              ctypes.c_size_t(nocc))
print(index)

0


In [6]:
import itertools
norb = 8
nocc = 4
# occ_list = np.array([1,3,4,5], dtype=np.int32)
for comb in itertools.combinations(np.arange(norb, dtype=np.uintp), nocc):
    occ_list = np.array(comb)
    rank = libhci.rank(occ_list.ctypes.data_as(ctypes.c_void_p),
                       rank_table.ctypes.data_as(ctypes.c_void_p),
                       ctypes.c_size_t(norb),
                       ctypes.c_size_t(nocc))
    print(occ_list, rank)

[0 1 2 3] 0
[0 1 2 4] 1
[0 1 2 5] 5
[0 1 2 6] 15
[0 1 2 7] 35
[0 1 3 4] 2
[0 1 3 5] 6
[0 1 3 6] 16
[0 1 3 7] 36
[0 1 4 5] 9
[0 1 4 6] 19
[0 1 4 7] 39
[0 1 5 6] 25
[0 1 5 7] 45
[0 1 6 7] 55
[0 2 3 4] 3
[0 2 3 5] 7
[0 2 3 6] 17
[0 2 3 7] 37
[0 2 4 5] 10
[0 2 4 6] 20
[0 2 4 7] 40
[0 2 5 6] 26
[0 2 5 7] 46
[0 2 6 7] 56
[0 3 4 5] 12
[0 3 4 6] 22
[0 3 4 7] 42
[0 3 5 6] 28
[0 3 5 7] 48
[0 3 6 7] 58
[0 4 5 6] 31
[0 4 5 7] 51
[0 4 6 7] 61
[0 5 6 7] 65
[1 2 3 4] 4
[1 2 3 5] 8
[1 2 3 6] 18
[1 2 3 7] 38
[1 2 4 5] 11
[1 2 4 6] 21
[1 2 4 7] 41
[1 2 5 6] 27
[1 2 5 7] 47
[1 2 6 7] 57
[1 3 4 5] 13
[1 3 4 6] 23
[1 3 4 7] 43
[1 3 5 6] 29
[1 3 5 7] 49
[1 3 6 7] 59
[1 4 5 6] 32
[1 4 5 7] 52
[1 4 6 7] 62
[1 5 6 7] 66
[2 3 4 5] 14
[2 3 4 6] 24
[2 3 4 7] 44
[2 3 5 6] 30
[2 3 5 7] 50
[2 3 6 7] 60
[2 4 5 6] 33
[2 4 5 7] 53
[2 4 6 7] 63
[2 5 6 7] 67
[3 4 5 6] 34
[3 4 5 7] 54
[3 4 6 7] 64
[3 5 6 7] 68
[4 5 6 7] 69


In [9]:
norb = 8
nocc = 4

rank_table = np.empty((nocc, norb-nocc+1), dtype=np.uint64)
libhci.get_rank_table(rank_table.ctypes.data_as(ctypes.c_void_p),
                                       ctypes.c_size_t(norb),
                                       ctypes.c_size_t(nocc))
print(rank_table)

rank = 57
occ_list = np.zeros(nocc, dtype=np.uintp)
libhci.unrank(ctypes.c_uint64(rank),
              occ_list.ctypes.data_as(ctypes.c_void_p),
              rank_table.ctypes.data_as(ctypes.c_void_p),
              ctypes.c_size_t(norb),
              ctypes.c_size_t(nocc))
print(occ_list)

[[ 0  1  2  3  4]
 [ 0  1  3  6 10]
 [ 0  1  4 10 20]
 [ 0  1  5 15 35]]
[1 2 6 7]


In [13]:
norb = 8
nocc = 4
nij = norb*(norb+1)//2
mock_eri_s8 = rng.random(nij*(nij+1)//2)

rank_table = np.empty((nocc, norb-nocc+1), dtype=np.uint64)
libhci.get_rank_table(rank_table.ctypes.data_as(ctypes.c_void_p),
                                       ctypes.c_size_t(norb),
                                       ctypes.c_size_t(nocc))
print(rank_table)

index_table = np.empty((nocc, norb-4+1), dtype=np.uint64)
libhci.get_rank_table(index_table.ctypes.data_as(ctypes.c_void_p),
                                       ctypes.c_size_t(norb),
                                       ctypes.c_size_t(4))
print(index_table)

double_excitations_aaaa = np.empty(math.comb(norb, 4), dtype=lib.double_excitation_entry)
libhci.load_from_eri(double_excitations_aaaa.ctypes.data_as(ctypes.c_void_p),
                     mock_eri_s8.ctypes.data_as(ctypes.c_void_p),
                     index_table.ctypes.data_as(ctypes.c_void_p),
                     ctypes.c_size_t(norb))
print(double_excitations_aaaa)

[[ 0  1  2  3  4]
 [ 0  1  3  6 10]
 [ 0  1  4 10 20]
 [ 0  1  5 15 35]]
[[ 0  1  2  3  4]
 [ 0  1  3  6 10]
 [ 0  1  4 10 20]
 [ 0  1  5 15 35]]
[( 0, 0.45399223, 0.97739602) ( 1, 0.66456192, 0.2903394 )
 ( 2, 0.75110648, 0.8318366 ) ( 3, 0.70147096, 0.26025527)
 ( 4, 0.26025527, 0.30156617) ( 5, 0.97739602, 0.75110648)
 ( 6, 0.38678769, 0.44416524) ( 7, 0.8318366 , 0.30156617)
 ( 8, 0.18729375, 0.5860672 ) ( 9, 0.26025527, 0.35105901)
 (10, 0.8189465 , 0.86301733) (11, 0.30156617, 0.15028426)
 (12, 0.30156617, 0.23543413) (13, 0.23543413, 0.01007403)
 (14, 0.35111116, 0.25355829) (15, 0.2903394 , 0.38678769)
 (16, 0.70147096, 0.26025527) (17, 0.44416524, 0.5860672 )
 (18, 0.8189465 , 0.15028426) (19, 0.18729375, 0.30156617)
 (20, 0.25147519, 0.15028426) (21, 0.86301733, 0.62689418)
 (22, 0.86301733, 0.01007403) (23, 0.07838165, 0.68665598)
 (24, 0.27230376, 0.48305479) (25, 0.86301733, 0.5860672 )
 (26, 0.44534544, 0.48186851) (27, 0.5860672 , 0.35111116)
 (28, 0.5860672 , 0.52385419

In [14]:
magnitudes = np.empty(math.comb(norb, 4), dtype=np.double)
libhci.get_max_magnitudes(double_excitations_aaaa.ctypes.data_as(ctypes.c_void_p),
                          magnitudes.ctypes.data_as(ctypes.c_void_p),
                          ctypes.c_size_t(math.comb(norb,4)))
print(magnitudes)

[1.43138825 0.95490133 1.58294308 0.96172623 0.56182144 1.7285025
 0.83095294 1.13340277 0.77336095 0.61131428 1.68196383 0.45185043
 0.5370003  0.24550816 0.60466945 0.6771271  0.96172623 1.03023244
 0.96923076 0.48885992 0.40175945 1.48991151 0.87309136 0.76503764
 0.75535855 1.44908453 0.92721395 0.93717836 1.10992139 0.3760649
 0.9098485  0.36118519 0.83462684 1.22269986 0.6734003  1.45257744
 1.01913035 0.41053953 0.7333437  1.68196383 0.97795319 1.02535781
 1.36090263 0.83962549 1.43293493 0.8245309  0.58654529 0.32530424
 0.51627513 0.85632432 1.18988381 1.00736987 1.14299478 1.54218654
 0.89141365 1.10876269 0.28237779 0.3218933  0.48844259 0.78945421
 1.23007208 0.86167597 0.92121139 0.65670874 1.2603617  1.15898853
 0.9371221  0.4054812  0.79063353 0.98682543]


In [15]:
print(magnitudes-(double_excitations_aaaa['ijkl']+double_excitations_aaaa['iljk']))

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
